# ATLAS — Multimodal Training

Boilerplate for feature-level late fusion. **All inputs are cleaned, preprocessed, and null-handled.** Fill in the TODO cells.

| Component | Owner | Output |
| --- | --- | --- |
| S3 image tensors → ResNet-18 | Kevin | `[B, 512]` |
| Tabular features → MLP | Swati | `[B, 64]` |
| USGS DEM → SmallCNN | Abhi | `[B, 64]` |
| Fusion, training, evaluation, XAI | Team — owner TBD | `[B, 640]` → logits |

`B` = batch size. Concatenation order: **image, tabular, DEM**.

## 1. Setup
**Owner: Team — TBD**

Set imports, device, seed, S3 paths, class names, and training hyperparameters.

In [ ]:

# Section 1 — Shared multimodal configuration
import io
import random
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
N_CLASSES = 3
CLASS_NAMES = ("Small", "Medium", "Large")
N_FOLDS = 5

IMAGE_EMB_DIM = 512
TAB_EMB_DIM = 64
DEM_EMB_DIM = 64
FUSION_DIM = IMAGE_EMB_DIM + TAB_EMB_DIM + DEM_EMB_DIM

BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

AWS_PROFILE = "ATLAS"
AWS_REGION = "us-east-2"
S3_BUCKET = "atlas-acm-research-f26"

IMAGE_PREFIX = "satellite/processed/"
DEM_PREFIX = "usgs/naip_aligned_v2/dem/"

TABULAR_KEY = (
    "dataset/atlas_wri_13score_v1/"
    "im3_atlas_wri_13scores_normalized_0to1.csv"
)

SPATIAL_FOLDS_KEY = (
    "artifacts/usgs_terrain_baselines/folds/v1/"
    "usgs_spatial_folds_v1.csv"
)

DEM_CHECKPOINT_PREFIX = (
    "artifacts/usgs_terrain_baselines/v1/cnn/"
)

# Resolve the repository root whether the notebook runs
# from ATLAS/ or ATLAS/models/.
cwd = Path.cwd().resolve()

if (cwd / "models" / "train_multimodal.ipynb").is_file():
    PROJECT_ROOT = cwd
elif (cwd / "train_multimodal.ipynb").is_file() and cwd.name == "models":
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        f"Cannot locate ATLAS repository root from {cwd}"
    )

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "multimodal_spatial_v1"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

s3_shared = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION
).client("s3")

print("Device:", device)
print("Fusion dimension:", FUSION_DIM)
print("S3 bucket:", S3_BUCKET)
print("Spatial folds:", N_FOLDS)


## 2. Load aligned inputs and split assignments
**Owner: Team — TBD**

Load the prepared facility manifest and shared train/validation/test assignments. Use facility IDs to retrieve matching image, tabular, DEM, and target inputs.

In [ ]:

# Section 2 — Shared facility manifest and spatial folds

def read_shared_s3_csv(key):
    response = s3_shared.get_object(
        Bucket=S3_BUCKET,
        Key=key
    )
    return pd.read_csv(
        io.BytesIO(response["Body"].read()),
        dtype={"id": str}
    )


def normalize_facility_ids(series):
    ids = series.astype("string").str.strip()
    assert ids.notna().all(), "Missing facility IDs"
    assert ids.str.fullmatch(r"\d{1,10}").all(), (
        "Facility IDs must contain 1–10 digits"
    )
    return ids.str.zfill(10)


spatial_folds = read_shared_s3_csv(SPATIAL_FOLDS_KEY)
tabular_source = read_shared_s3_csv(TABULAR_KEY)

required_fold_columns = {
    "id", "lat", "lon", "size_class", "geo_group", "fold"
}
assert required_fold_columns.issubset(spatial_folds.columns)

required_tabular_columns = {"id", "size_class"}
assert required_tabular_columns.issubset(tabular_source.columns)

spatial_folds["id"] = normalize_facility_ids(
    spatial_folds["id"]
)
tabular_source["id"] = normalize_facility_ids(
    tabular_source["id"]
)

assert spatial_folds["id"].is_unique
assert tabular_source["id"].is_unique
assert len(spatial_folds) == 180, (
    f"Expected 180 facilities, got {len(spatial_folds)}"
)

assert set(spatial_folds["fold"]) == set(range(1, N_FOLDS + 1))

manifest = spatial_folds[
    ["id", "lat", "lon", "size_class", "geo_group", "fold"]
].copy()

tab_labels = tabular_source[
    ["id", "size_class"]
].rename(columns={"size_class": "tabular_size_class"})

manifest = manifest.merge(
    tab_labels,
    on="id",
    how="left",
    validate="one_to_one"
)

assert manifest["tabular_size_class"].notna().all(), (
    "Some facilities are missing from tabular data"
)

assert (
    pd.to_numeric(manifest["size_class"])
    == pd.to_numeric(manifest["tabular_size_class"])
).all(), "Spatial-fold and tabular labels disagree"

manifest = manifest.drop(columns=["tabular_size_class"])

manifest["image_key"] = (
    IMAGE_PREFIX + manifest["id"] + ".pt"
)
manifest["dem_key"] = (
    DEM_PREFIX + manifest["id"] + "_dem.tif"
)

assert manifest["id"].is_unique
assert manifest["size_class"].notna().all()
assert manifest["geo_group"].notna().all()

manifest["size_class"] = pd.to_numeric(
    manifest["size_class"]
).astype(int)

assert manifest["size_class"].isin(
    range(N_CLASSES)
).all()

assert (
    manifest.groupby("geo_group")["fold"].nunique() == 1
).all(), "Geographic group appears in multiple folds"

manifest = manifest.sort_values("id").reset_index(drop=True)

# Section 6 will use this shared dataframe.
MULTIMODAL_MANIFEST = manifest

print("Multimodal manifest:", MULTIMODAL_MANIFEST.shape)
print("Unique facilities:", MULTIMODAL_MANIFEST["id"].nunique())
print("Spatial fold counts:")
print(MULTIMODAL_MANIFEST["fold"].value_counts().sort_index())
print("Class distribution:")
print(MULTIMODAL_MANIFEST["size_class"].value_counts().sort_index())


# temp check section

In [ ]:
fold_class_counts = pd.crosstab(
    MULTIMODAL_MANIFEST["fold"],
    MULTIMODAL_MANIFEST["size_class"]
)

fold_class_counts = fold_class_counts.reindex(
    index=range(1, N_FOLDS + 1),
    columns=range(N_CLASSES),
    fill_value=0,
)

fold_class_counts.columns = CLASS_NAMES

print("Class distribution by geographic fold:")
print(fold_class_counts)

print("\nFolds missing Small facilities:")
print(fold_class_counts.index[
    fold_class_counts["Small"] == 0
].tolist())

In [ ]:

# Section 2b — Rotating geographic train/val/test splits

def make_spatial_split(manifest, test_fold):
    """Use one outer test fold, the next fold for validation."""
    if test_fold not in range(1, N_FOLDS + 1):
        raise ValueError(f"Invalid test fold: {test_fold}")

    val_fold = (test_fold % N_FOLDS) + 1

    result = manifest.copy()
    result["split"] = "train"
    result.loc[result["fold"] == val_fold, "split"] = "val"
    result.loc[result["fold"] == test_fold, "split"] = "test"

    assert result["id"].is_unique
    assert set(result["split"]) == {"train", "val", "test"}

    # No geographic group may cross a split boundary.
    assert (
        result.groupby("geo_group")["split"].nunique() == 1
    ).all(), "Geographic leakage detected"

    for split in ("train", "val", "test"):
        subset = result.loc[result["split"] == split]
        assert set(subset["size_class"]) == set(range(N_CLASSES)), (
            f"{split} partition is missing a class"
        )

    return result


# Initial integration run: fold 1 held out for testing.
TEST_FOLD = 1
MULTIMODAL_SPLIT = make_spatial_split(
    MULTIMODAL_MANIFEST,
    test_fold=TEST_FOLD
)

print("\nInitial geographic split:")
print(
    pd.crosstab(
        MULTIMODAL_SPLIT["split"],
        MULTIMODAL_SPLIT["size_class"]
    ).reindex(["train", "val", "test"])
)

print("\nPartition sizes:")
print(
    MULTIMODAL_SPLIT["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)


## 3. Image encoder — Kevin

Load prepared image tensors from S3 (`satellite/processed/`). Each `.pt` contains `id`, `image`, and `source_key`.

**`[B, 3, 224, 224]` → ResNet-18 → `[B, 512]`**

Extract features before the final classification layer. Specify checkpoint and frozen/trainable layers.

In [ ]:
from io import BytesIO
import boto3
import torch
from torch import nn
from torchvision import models, transforms

BUCKET="atlas-acm-research-f26"
IMAGE_PREFIX="satellite/processed/"
#could change to pretrained weights if wanted 
IMAGE_CHECKPOINT_KEY = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
s3 = boto3.Session(
    profile_name="ATLAS",
    region_name="us-east-2",
).client("s3")

image_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

def read_s3_bytes(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    try:
        return body.read()
    finally:
        body.close()
        
def load_image_tensor(s3_key):
    payload = torch.load(
        BytesIO(read_s3_bytes(s3_key)),
        map_location="cpu",
        weights_only=True,
    )
    image = payload["image"].float()

    if image.shape != (3, 224, 224):
        raise ValueError(
            f"Unexpected image shape for {s3_key}: {tuple(image.shape)}"
        )

    #stored tensors are RGB values in [0, 1].
    #apply normalization once, matching the visual baseline.
    return image_normalize(image)

class ImageEncoder(nn.Module):
    output_dim = 512

    def __init__(self, checkpoint_key=None, freeze_backbone=True):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if checkpoint_key is None
            else None
        )
        self.backbone = models.resnet18(weights=weights)

        if checkpoint_key is not None:
            self.backbone.fc = nn.Linear(
                self.backbone.fc.in_features, 3
            )
            state_dict = torch.load(
                BytesIO(read_s3_bytes(checkpoint_key)),
                map_location="cpu",
                weights_only=True,
            )
            self.backbone.load_state_dict(state_dict)

        #return image features instead of Small/Medium/Large classification
        self.backbone.fc = nn.Identity()

        self.freeze_backbone = freeze_backbone
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze_backbone

        if freeze_backbone:
            self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)

        #preserve frozen BatchNorm statistics during fused training.
        if self.freeze_backbone:
            self.backbone.eval()

        return self

    def forward(self, images):
        #input: normalized [B, 3, 224, 224]
        #output: [B, 512]
        return self.backbone(images)


image_encoder = ImageEncoder(
    checkpoint_key=IMAGE_CHECKPOINT_KEY,
    freeze_backbone=True,
).to(device)

print(f"Image encoder device: {device}")
print("Image embedding dimension:", image_encoder.output_dim)    

## 4. Tabular encoder — Swati

Load prepared IM3 + EIA + WRI features and adapt `TabularEncoder` for fusion.

**`[B, F]` → MLP → `[B, 64]`**

`F` is the prepared feature count. Return embeddings; train with the fused classification loss.

In [ ]:
# TODO (Swati): Load prepared tabular features and define the MLP encoder.
# [B, F] -> [B, 64]

In [ ]:
# Tabular config + load — shared multimodal source
# Uses Section 2's verified S3 tabular data and facility IDs.

TAB_EMB_DIM = 64

TAB_ID_COLS = ["id", "name", "ref", "operator", "county", "state", "state_abb", "state_id", "county_id",
               "county_fips", "pfaf_id", "retail_period", "period"]
TAB_LOCATION_COLS = ["lat", "lon"]          # so the model can't just memorize location
TAB_LEAKAGE_COLS = ["sqft", "size_class"]   # size_class is built from sqft -> leaks the size answer
TAB_DROP_COLS = TAB_ID_COLS + TAB_LOCATION_COLS + TAB_LEAKAGE_COLS

# Restrict tabular records to facilities with aligned modalities.
shared_ids = MULTIMODAL_MANIFEST["id"].tolist()

tab_df = tabular_source.loc[
    tabular_source["id"].isin(shared_ids)
].copy()

assert tab_df["id"].is_unique
assert len(tab_df) == len(shared_ids) == 180
assert set(tab_df["id"]) == set(shared_ids)

# Preserve manifest ordering.
tab_df = (
    tab_df.set_index("id")
    .loc[shared_ids]
    .reset_index()
)

assert tab_df["id"].tolist() == shared_ids

print("Aligned tabular records:", tab_df.shape)


In [ ]:
# Split assignment — shared geographic train/val/test
# All modalities use the same facility assignments.

split_lookup = MULTIMODAL_SPLIT[
    ["id", "split", "size_class"]
].rename(columns={"size_class": "shared_size_class"})

tab_df = tab_df.merge(
    split_lookup,
    on="id",
    how="left",
    validate="one_to_one",
)

assert tab_df["split"].notna().all()
assert tab_df["shared_size_class"].notna().all()

assert (
    pd.to_numeric(tab_df["size_class"])
    == pd.to_numeric(tab_df["shared_size_class"])
).all(), "Tabular and shared labels disagree"

tab_df = tab_df.drop(columns=["shared_size_class"])

tab_train_ids = tab_df.loc[
    tab_df["split"] == "train", "id"
].tolist()

tab_val_ids = tab_df.loc[
    tab_df["split"] == "val", "id"
].tolist()

tab_test_ids = tab_df.loc[
    tab_df["split"] == "test", "id"
].tolist()

assert len(tab_train_ids) == 107
assert len(tab_val_ids) == 37
assert len(tab_test_ids) == 36

assert not (
    set(tab_train_ids) & set(tab_val_ids)
    or set(tab_train_ids) & set(tab_test_ids)
    or set(tab_val_ids) & set(tab_test_ids)
)

print(
    "Train / val / test:",
    len(tab_train_ids),
    len(tab_val_ids),
    len(tab_test_ids),
)


In [ ]:
# Pick features + preprocess (impute/scale numeric, one-hot categorical). FIT ON TRAIN ONLY.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

tab_feat_df = tab_df.drop(columns=[c for c in TAB_DROP_COLS + ["split"] if c in tab_df.columns])
train_mask = (tab_df["split"] == "train").values

# drop columns that are constant in train (no information)
tab_constant = [c for c in tab_feat_df.columns if tab_feat_df.loc[train_mask, c].nunique(dropna=False) <= 1]
tab_feat_df = tab_feat_df.drop(columns=tab_constant)

tab_num_cols = tab_feat_df.select_dtypes(include=np.number).columns.tolist()
tab_cat_cols = [c for c in tab_feat_df.columns if c not in tab_num_cols]
for c in TAB_LEAKAGE_COLS + TAB_LOCATION_COLS:
    assert c not in tab_feat_df.columns, f"{c} is still a feature"

tab_preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), tab_num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), tab_cat_cols),
])
tab_preprocess.fit(tab_feat_df.loc[train_mask])                      # train rows only
tab_X = tab_preprocess.transform(tab_feat_df).astype(np.float32)     # applied to every row
assert not np.isnan(tab_X).any(), "NaNs left after preprocessing"

TAB_FEATURE_NAMES = list(tab_preprocess.get_feature_names_out())
TAB_IN_DIM = tab_X.shape[1]                                          # F, never hardcoded

# id -> row lookup, used by the Dataset in section 6
tab_row_of = {i: r for r, i in enumerate(tab_df["id"])}

def get_tabular(ids):
    """Tabular feature tensor [B, F] for a list of facility ids (same order as ids)."""
    return torch.from_numpy(tab_X[[tab_row_of[i] for i in ids]])

print("Dropped (ids/location/leakage):", [c for c in TAB_DROP_COLS if c in tab_df.columns])
print("Dropped constant:", tab_constant)
print(f"Numeric ({len(tab_num_cols)}):", tab_num_cols)
print(f"Categorical ({len(tab_cat_cols)}):", tab_cat_cols)
print("F =", TAB_IN_DIM)

In [ ]:
# 4d. Tabular MLP encoder: [B, F] -> [B, 64]. Returns embeddings only (no head);
class TabularEncoder(nn.Module):
    def __init__(self, in_dim, emb_dim=TAB_EMB_DIM, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, emb_dim), nn.LayerNorm(emb_dim),
        )
    def forward(self, x):
        return self.net(x)

# shape check on a few training facilities
_enc = TabularEncoder(TAB_IN_DIM).eval()
with torch.no_grad():
    _out = _enc(get_tabular(tab_train_ids[:8]))
assert _out.shape == (8, TAB_EMB_DIM), _out.shape
print(f"TabularEncoder OK: [8, {TAB_IN_DIM}] -> {list(_out.shape)}")
del _enc, _out

## 5. DEM encoder — Abhi

Load prepared USGS DEM tensors and implement SmallCNN.

**`[B, C_dem, H_dem, W_dem]` → SmallCNN → `[B, 64]`**

Return embeddings before any branch classification head.

In [ ]:

# USGS DEM encoder — Abhi
# Pretrained spatial-fold CNN; output: [B, 64].
# No checkpoint is selected automatically: splits must be aligned first.

from pathlib import Path
import numpy as np
import rasterio
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset

DEM_TARGET_SHAPE = (64, 96)


class TerrainCNN(nn.Module):
    def __init__(self, embedding_dim=64, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.embedding = nn.Sequential(
            nn.Linear(64, embedding_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def encode(self, image, mask):
        if image.ndim != 4 or image.shape[1] != 1:
            raise ValueError("Expected DEM tensor [B, 1, H, W]")
        if mask.shape != image.shape:
            raise ValueError("DEM and mask shapes must match")

        x = self.features(image * mask)
        m = F.adaptive_avg_pool2d(mask, x.shape[-2:])
        pooled = (
            (x * m).sum(dim=(2, 3))
            / m.sum(dim=(2, 3)).clamp_min(1e-6)
        )
        return self.embedding(pooled)

    def forward(self, image, mask):
        return self.classifier(self.encode(image, mask))


class USGSDemDataset(Dataset):
    """DEM-only dataset, aligned by facility ID.

    Input dataframe requires an 'id' column.
    Returns image, mask, and id; labels remain owned by
    the shared multimodal dataset.
    """
    def __init__(self, dataframe, dem_dir, scale,
                 target_shape=DEM_TARGET_SHAPE):
        self.df = dataframe.reset_index(drop=True).copy()
        self.dem_dir = Path(dem_dir)
        self.scale = float(scale)
        self.target_shape = tuple(target_shape)

        if "id" not in self.df.columns:
            raise ValueError("Dataframe requires an id column")
        if not np.isfinite(self.scale) or self.scale <= 0:
            raise ValueError("scale must be positive and finite")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        facility_id = str(self.df.iloc[idx]["id"])
        path = self.dem_dir / f"{facility_id}_dem.tif"

        with rasterio.open(path) as src:
            dem = src.read(1, masked=True)

        values = dem.compressed()
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(f"Invalid DEM: {facility_id}")

        h, w = dem.shape
        H, W = self.target_shape
        if h > H or w > W:
            raise ValueError(
                f"{facility_id}: {h}x{w} exceeds {H}x{W}"
            )

        mean = float(values.mean())
        centered = np.asarray(
            (dem.filled(mean) - mean) / self.scale,
            dtype=np.float32
        )

        image = np.zeros((H, W), dtype=np.float32)
        mask = np.zeros((H, W), dtype=np.float32)
        top, left = (H - h) // 2, (W - w) // 2

        image[top:top+h, left:left+w] = centered
        mask[top:top+h, left:left+w] = (
            ~np.ma.getmaskarray(dem)
        ).astype(np.float32)

        return {
            "image": torch.from_numpy(image[None]),
            "mask": torch.from_numpy(mask[None]),
            "id": facility_id
        }


def fit_dem_scale(train_df, dem_dir):
    """Fit normalization on training facilities only."""
    sums, count = 0.0, 0
    for fid in train_df["id"]:
        with rasterio.open(
            Path(dem_dir) / f"{fid}_dem.tif"
        ) as src:
            dem = src.read(1, masked=True)

        values = dem.compressed().astype(np.float64)
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(f"Invalid DEM: {fid}")

        sums += np.square(values - values.mean()).sum()
        count += len(values)

    if count == 0:
        raise ValueError("No valid training DEM pixels")

    scale = float(np.sqrt(sums / count))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid fitted DEM scale")
    return scale


def load_terrain_encoder(checkpoint_path, device="cpu",
                         freeze=True, expected_fold=None):
    """Load a pretrained CNN with strict architecture validation."""
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    config = checkpoint["config"]
    if config["embedding_dim"] != 64:
        raise ValueError("Expected 64-dimensional embedding")
    if tuple(config["target_shape"]) != DEM_TARGET_SHAPE:
        raise ValueError("Unexpected DEM target shape")

    if expected_fold is not None:
        if int(checkpoint["fold"]) != int(expected_fold):
            raise ValueError("Checkpoint fold mismatch")

    scale = float(checkpoint["train_scale_m"])
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid checkpoint normalization scale")

    encoder = TerrainCNN(embedding_dim=64)
    encoder.load_state_dict(
        checkpoint["model_state_dict"], strict=True
    )
    encoder.to(device)

    if freeze:
        for parameter in encoder.parameters():
            parameter.requires_grad_(False)
        encoder.eval()

    return encoder, scale


# Interface for the future fusion section:
#
# encoder, scale = load_terrain_encoder(
#     checkpoint_path,
#     device=device,
#     freeze=True,
#     expected_fold=fold,
# )
#
# dem_dataset = USGSDemDataset(facilities, dem_dir, scale)
# dem_embedding = encoder.encode(dem_image, dem_mask)
# assert dem_embedding.shape == (batch_size, 64)
#
# Do not reuse a pretrained fold checkpoint across arbitrary
# multimodal splits. Confirm facility-level split compatibility.


## 6. Dataset and DataLoaders
**Owner: Team — TBD**

Each sample returns image, tabular, DEM, label, and facility ID. Build train, validation, and test loaders with matching modality order.

In [ ]:
# TODO: Multimodal Dataset and train/validation/test DataLoaders.

In [ ]:

# Section 6 — Shared multimodal Dataset and DataLoaders

DEM_LOCAL_DIR = PROJECT_ROOT / "data/temp/usgs_naip_aligned_v2"


class MultimodalDataset(Dataset):
    """Facility-ID-aligned NAIP, tabular, DEM, and labels."""

    def __init__(self, dataframe, dem_dir, dem_scale,
                 tabular_lookup=None):
        self.df = dataframe.reset_index(drop=True).copy()

        assert self.df["id"].is_unique
        assert self.df["size_class"].isin(range(N_CLASSES)).all()

        self.dem_dataset = USGSDemDataset(
            self.df,
            dem_dir=dem_dir,
            scale=dem_scale,
        )

        # Explicit dependency on Swati's fitted preprocessing.
        self.tabular_lookup = tabular_lookup

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        facility_id = str(row["id"])

        image = load_image_tensor(row["image_key"])
        dem_sample = self.dem_dataset[index]

        assert dem_sample["id"] == facility_id

        sample = {
            "id": facility_id,
            "image": image,
            "dem": dem_sample["image"],
            "dem_mask": dem_sample["mask"],
            "label": torch.tensor(
                int(row["size_class"]),
                dtype=torch.long,
            ),
        }

        if self.tabular_lookup is not None:
            tabular = self.tabular_lookup([facility_id])
            assert tabular.ndim == 2 and tabular.shape[0] == 1
            sample["tabular"] = tabular[0].float()

        return sample


def build_multimodal_loaders(
    split_df,
    dem_dir,
    tabular_lookup,
    batch_size=BATCH_SIZE,
    num_workers=0,
):
    """Build geographically separated loaders for one outer fold."""

    if tabular_lookup is None:
        raise ValueError(
            "Tabular preprocessing must be fitted before "
            "building full multimodal DataLoaders."
        )

    dem_dir = Path(dem_dir)
    assert dem_dir.is_dir(), f"Missing DEM directory: {dem_dir}"

    train_df = split_df.loc[
        split_df["split"] == "train"
    ].copy()

    # Fit normalization using training facilities only.
    dem_scale = fit_dem_scale(train_df, dem_dir)

    loaders = {}

    for split in ("train", "val", "test"):
        subset = split_df.loc[
            split_df["split"] == split
        ].copy()

        dataset = MultimodalDataset(
            dataframe=subset,
            dem_dir=dem_dir,
            dem_scale=dem_scale,
            tabular_lookup=tabular_lookup,
        )

        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
        )

    return loaders, dem_scale


print("MultimodalDataset defined.")
print("DataLoader factory defined.")
print("Local DEM directory:", DEM_LOCAL_DIR)


In [ ]:
print("fit_dem_scale available:", "fit_dem_scale" in globals())
print("USGSDemDataset available:", "USGSDemDataset" in globals())
print("DEM directory exists:", DEM_LOCAL_DIR.is_dir())

assert callable(fit_dem_scale)
assert DEM_LOCAL_DIR.is_dir()

In [ ]:
assert DEM_LOCAL_DIR.is_dir()

missing_dem = [
    fid
    for fid in MULTIMODAL_SPLIT["id"]
    if not (DEM_LOCAL_DIR / f"{fid}_dem.tif").is_file()
]

print("Missing local DEMs:", len(missing_dem))
assert not missing_dem, missing_dem[:10]

train_df = MULTIMODAL_SPLIT.loc[
    MULTIMODAL_SPLIT["split"] == "train"
]

dem_scale = fit_dem_scale(train_df, DEM_LOCAL_DIR)

print("Train-only DEM scale:", dem_scale)
assert np.isfinite(dem_scale) and dem_scale > 0

In [ ]:
# Test one real facility without requiring tabular features.

sample_df = MULTIMODAL_SPLIT.loc[
    MULTIMODAL_SPLIT["split"] == "train"
].head(1)

test_dataset = MultimodalDataset(
    dataframe=sample_df,
    dem_dir=DEM_LOCAL_DIR,
    dem_scale=dem_scale,
    tabular_lookup=None,
)

sample = test_dataset[0]

print("Facility ID:", sample["id"])
print("NAIP shape:", tuple(sample["image"].shape))
print("DEM shape:", tuple(sample["dem"].shape))
print("DEM mask shape:", tuple(sample["dem_mask"].shape))
print("Label:", sample["label"].item())

assert sample["image"].shape == (3, 224, 224)
assert sample["dem"].shape == (1, 64, 96)
assert sample["dem_mask"].shape == (1, 64, 96)
assert torch.isfinite(sample["image"]).all()
assert torch.isfinite(sample["dem"]).all()

print("NAIP + DEM alignment: PASSED")

## 7. Feature concatenation and classifier
**Owner: Team — TBD**

Concatenate along `dim=1`, in image/tabular/DEM order:

**`[B, 512] + [B, 64] + [B, 64] → [B, 640] → classifier → [B, K]`**

`K` = number of classes. Return logits for cross-entropy loss.

In [ ]:
# TODO: Concatenate image, tabular, DEM embeddings and define the classifier.
# [B, 640] -> [B, K] logits

In [ ]:

# Section 7 — Late-fusion classifier
# Concatenation order: image, tabular, DEM.

class MultimodalFusion(nn.Module):
    def __init__(
        self,
        image_encoder,
        tabular_encoder,
        dem_encoder,
        hidden_dim=128,
        dropout=0.3,
    ):
        super().__init__()

        self.image_encoder = image_encoder
        self.tabular_encoder = tabular_encoder
        self.dem_encoder = dem_encoder

        self.classifier = nn.Sequential(
            nn.LayerNorm(FUSION_DIM),
            nn.Linear(FUSION_DIM, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, N_CLASSES),
        )

    def forward(self, images, tabular, dem, dem_mask):
        image_features = self.image_encoder(images)
        tabular_features = self.tabular_encoder(tabular)
        dem_features = self.dem_encoder.encode(dem, dem_mask)

        assert image_features.shape[1] == IMAGE_EMB_DIM
        assert tabular_features.shape[1] == TAB_EMB_DIM
        assert dem_features.shape[1] == DEM_EMB_DIM

        fused = torch.cat(
            [image_features, tabular_features, dem_features],
            dim=1,
        )

        return self.classifier(fused)


print("Fusion architecture defined.")
print("Expected input dimension:", FUSION_DIM)
print("Output classes:", N_CLASSES)


## 8. Integration check
**Owner: Team — TBD**

Run one batch. Verify embedding/logit shapes, compute loss, and confirm gradients reach all intended trainable components.

In [ ]:
# TODO: One-batch forward pass, loss, and gradient checks.

## 9. Training
**Owner: Team — TBD**

Define loss, optimizer, scheduler, and epochs. Track train/validation loss and macro-F1; use validation for early stopping and best-checkpoint selection. Save weights and model configuration.

In [ ]:
# TODO: Training, validation, early stopping, and best-checkpoint saving.

## 10. Evaluation
**Owner: Team — TBD**

Evaluate the best checkpoint on held-out test facilities. Report accuracy, macro-F1, per-class precision/recall/F1, and a confusion matrix. Export predictions.

Compare on the same test facilities: image only, image + tabular, image + DEM, and all three modalities.

In [ ]:
# TODO: Test metrics, confusion matrix, predictions, and modality ablations.

## 11. XAI scope — proposed
**Owner: Team — TBD**

| Component | Method |
| --- | --- |
| Image | Grad-CAM on ResNet-18 |
| Tabular | Integrated Gradients or SHAP |
| DEM | Input attribution or SmallCNN Grad-CAM |
| Modalities | Ablation comparison |

Target the final fused prediction. Start with image Grad-CAM and modality ablations; add tabular/DEM explanations afterward. Show a few correct and incorrect test predictions.

In [ ]:
# TODO: XAI methods and example visualizations.

## 12. Results and handoff
**Owner: Team — TBD**

Record checkpoint, metrics, ablations, XAI examples, artifact paths, and remaining tasks by owner.

In [ ]:
# TODO: Experiment summary and artifact paths.